# 🧩 Mejora del Algoritmo A* con Weighted A* para Resolver el Cubo Rubik
**Proyecto Final – Especialización en Inteligencia Artificial**

---

## 📋 Instrucciones de Ejecución
1. Abrir en **Google Colab** (subir el archivo `.ipynb` o abrir desde Drive).
2. Ejecutar las celdas **en orden** con `Runtime → Run All` o `Ctrl+F9`.
3. **Sin instalaciones externas** — solo Python estándar + `matplotlib` y `numpy`, disponibles por defecto en Colab.
4. Al finalizar se generan gráficas comparativas automáticas.

> **Nota:** Se utiliza el Cubo 2×2 (Pocket Cube, ~3.7M estados) para comparaciones factibles en tiempo real.

## 1. 📦 Importaciones

In [ ]:
import heapq
import time
import random
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.gridspec import GridSpec
from IPython.display import display

print("✅ Librerías importadas correctamente.")

## 2. 🎨 Visualizador del Cubo (Estilo Desplegado en Cruz)

Dibujamos el cubo como un **desarrollo plano en cruz**, igual al estilo del notebook de referencia del curso, pero con colores reales del Cubo Rubik y bordes redondeados para máxima legibilidad.

```
        [  U  ]
[  L  ] [  F  ] [  R  ] [  B  ]
        [  D  ]
```

In [ ]:
# ══════════════════════════════════════════════════════════════
#  PALETA DE COLORES OFICIAL DEL CUBO RUBIK
# ══════════════════════════════════════════════════════════════
COLORS_HEX = {
    0: '#FFFFFF',  # Blanco  (cara U)
    1: '#009B48',  # Verde   (cara D)
    2: '#B71234',  # Rojo    (cara F)
    3: '#0046AD',  # Azul    (cara B)
    4: '#FF5800',  # Naranja (cara L)
    5: '#FFD500',  # Amarillo(cara R)
}
COLORS_NAME = {
    0: 'Blanco', 1: 'Verde', 2: 'Rojo',
    3: 'Azul',   4: 'Naranja', 5: 'Amarillo'
}

def draw_cube_2x2(state, ax=None, title="", sticker_size=0.9):
    """
    Dibuja el cubo 2x2 como desarrollo plano en cruz (estilo del curso).

    Layout:
            [ U ]
    [ L ]  [ F ]  [ R ]  [ B ]
            [ D ]

    Índices del estado (24 valores):
      U = state[0:4]   D = state[4:8]
      F = state[8:12]  B = state[12:16]
      L = state[16:20] R = state[20:24]

    Parámetros:
    -----------
    state       : tuple de 24 enteros (0-5)
    ax          : eje matplotlib (None = crea uno nuevo)
    title       : título sobre la figura
    sticker_size: tamaño de cada sticker en unidades de plot
    """
    standalone = ax is None
    if standalone:
        fig, ax = plt.subplots(figsize=(7, 5))
        fig.patch.set_facecolor('#1e1e2e')   # fondo oscuro elegante
        ax.set_facecolor('#1e1e2e')

    S = sticker_size        # tamaño del sticker
    PAD = 0.06              # padding interno del sticker (bordes redondeados)
    GAP = 0.12              # espacio entre faces
    SEP = 0.025             # separación entre stickers de la misma cara

    # Paso de una cara (2 stickers + separación)
    face_step = 2 * S + SEP + GAP

    # Datos de cada cara: (nombre, offset_x, offset_y, stickers)
    face_layout = [
        ('U', face_step,       face_step,  state[0:4]),
        ('L', 0,               0,          state[16:20]),
        ('F', face_step,       0,          state[8:12]),
        ('R', 2 * face_step,   0,          state[20:24]),
        ('B', 3 * face_step,   0,          state[12:16]),
        ('D', face_step,      -face_step,  state[4:8]),
    ]

    # Colores de texto adaptados al fondo del sticker
    def text_color_for(hex_color):
        r = int(hex_color[1:3], 16)
        g = int(hex_color[3:5], 16)
        b = int(hex_color[5:7], 16)
        luminance = 0.299*r + 0.587*g + 0.114*b
        return '#1a1a1a' if luminance > 128 else '#f0f0f0'

    for face_name, ox, oy, stickers in face_layout:
        # ── Sombra de la cara (fondo oscuro) ──────────────────
        face_w = 2*S + SEP
        shadow = patches.FancyBboxPatch(
            (ox - 0.05, oy - face_w - 0.05),
            face_w + 0.1, face_w + 0.1,
            boxstyle=f"round,pad=0.08",
            facecolor='#000000', edgecolor='none',
            alpha=0.4, zorder=1
        )
        ax.add_patch(shadow)

        # ── Marco exterior de la cara ──────────────────────────
        frame = patches.FancyBboxPatch(
            (ox - 0.04, oy - face_w - 0.04),
            face_w + 0.08, face_w + 0.08,
            boxstyle=f"round,pad=0.06",
            facecolor='#2a2a2a', edgecolor='#444444',
            linewidth=1.5, zorder=2
        )
        ax.add_patch(frame)

        # ── Stickers individuales ──────────────────────────────
        for idx, color_id in enumerate(stickers):
            col = idx % 2
            row = idx // 2
            x = ox + col * (S + SEP)
            y = oy - row * (S + SEP)

            hex_col = COLORS_HEX[color_id]

            # Brillo superior-izquierdo del sticker (efecto 3D)
            highlight = patches.FancyBboxPatch(
                (x + 0.02, y - S + 0.02), S - 0.04, S - 0.04,
                boxstyle=f"round,pad={PAD}",
                facecolor='#ffffff', edgecolor='none',
                alpha=0.18, zorder=3
            )
            ax.add_patch(highlight)

            # Sticker principal
            sticker = patches.FancyBboxPatch(
                (x, y - S), S, S,
                boxstyle=f"round,pad={PAD}",
                facecolor=hex_col,
                edgecolor='#111111',
                linewidth=1.4,
                zorder=4
            )
            ax.add_patch(sticker)

        # ── Etiqueta de la cara (centrada) ────────────────────
        cx = ox + S + SEP / 2
        cy = oy - S - SEP / 2
        ax.text(cx, cy, face_name,
                ha='center', va='center',
                fontsize=8, fontweight='bold',
                color='#aaaaaa', alpha=0.7, zorder=5)

    # ── Título ────────────────────────────────────────────────
    if title:
        ax.set_title(title, fontsize=11, fontweight='bold',
                     color='#e0e0e0', pad=10)

    # ── Ajuste de ejes ────────────────────────────────────────
    margin = 0.4
    ax.set_aspect('equal')
    ax.axis('off')
    ax.set_xlim(-margin, 4 * face_step + margin)
    ax.set_ylim(-2 * face_step - margin, 2 * face_step + margin)

    if standalone:
        plt.tight_layout()
        plt.show()


# ── Prueba rápida: mostrar el cubo resuelto ───────────────────
SOLVED_STATE = tuple([0]*4 + [1]*4 + [2]*4 + [3]*4 + [4]*4 + [5]*4)

print("Vista previa del cubo resuelto:")
draw_cube_2x2(SOLVED_STATE, title="Cubo Resuelto ✅")

## 3. 🔧 Representación y Movimientos del Cubo 2×2

In [ ]:
# ══════════════════════════════════════════════════════════════
#  ESTADO: tupla inmutable de 24 enteros (6 caras × 4 stickers)
#  Orden: U(0-3) D(4-7) F(8-11) B(12-15) L(16-19) R(20-23)
# ══════════════════════════════════════════════════════════════

def is_solved(state):
    """True si cada cara tiene los 4 stickers del mismo color."""
    return all(state[i*4] == state[i*4+j] for i in range(6) for j in range(4))

def apply_cycles(state, cycles):
    """
    Aplica una permutación cíclica al estado.
    Cada ciclo es una lista de índices que rotan en sentido horario.
    """
    state = list(state)
    for cycle in cycles:
        tmp = state[cycle[-1]]
        for i in range(len(cycle)-1, 0, -1):
            state[cycle[i]] = state[cycle[i-1]]
        state[cycle[0]] = tmp
    return tuple(state)

def inverse_move(state, cycles):
    """Movimiento inverso = aplicar 3 veces el movimiento directo."""
    s = apply_cycles(state, cycles)
    s = apply_cycles(s, cycles)
    return apply_cycles(s, cycles)

# ── Ciclos de cada movimiento ─────────────────────────────────
U_CY = [[0,2,3,1], [8,16,12,20], [9,17,13,21]]
D_CY = [[4,5,7,6], [10,22,14,18], [11,23,15,19]]
F_CY = [[8,9,11,10], [2,20,5,23], [3,18,4,21]]
B_CY = [[12,14,15,13], [0,22,7,16], [1,23,6,17]]
L_CY = [[16,17,19,18], [0,8,4,15], [2,10,6,13]]
R_CY = [[20,22,23,21], [1,14,5,9], [3,12,7,11]]

MOVES = {
    'U' : lambda s: apply_cycles(s, U_CY),
    "U'": lambda s: inverse_move(s, U_CY),
    'D' : lambda s: apply_cycles(s, D_CY),
    "D'": lambda s: inverse_move(s, D_CY),
    'F' : lambda s: apply_cycles(s, F_CY),
    "F'": lambda s: inverse_move(s, F_CY),
    'B' : lambda s: apply_cycles(s, B_CY),
    "B'": lambda s: inverse_move(s, B_CY),
    'L' : lambda s: apply_cycles(s, L_CY),
    "L'": lambda s: inverse_move(s, L_CY),
    'R' : lambda s: apply_cycles(s, R_CY),
    "R'": lambda s: inverse_move(s, R_CY),
}

# ── Verificación de correctitud ───────────────────────────────
assert MOVES["U'"](MOVES['U'](SOLVED_STATE)) == SOLVED_STATE, "Error en U/U'"
assert MOVES["R'"](MOVES['R'](SOLVED_STATE)) == SOLVED_STATE, "Error en R/R'"
assert is_solved(SOLVED_STATE), "Error: estado inicial debería estar resuelto"

print(f"✅ {len(MOVES)} movimientos definidos y verificados.")

## 4. 🎲 Generador de Estados Revueltos

In [ ]:
def scramble_cube(n_moves=5, seed=None):
    """
    Aplica n_moves movimientos aleatorios desde el estado resuelto.
    Retorna (estado_revuelto, lista_de_movimientos).
    Usando seed fijo los experimentos son reproducibles.
    """
    if seed is not None:
        random.seed(seed)
    state = SOLVED_STATE
    names = list(MOVES.keys())
    applied = []
    for _ in range(n_moves):
        # Evitar mover la misma cara dos veces seguidas
        valid = [m for m in names if not applied or m[0] != applied[-1][0]]
        move = random.choice(valid)
        state = MOVES[move](state)
        applied.append(move)
    return state, applied

# ── Visualizar cubo antes y después del scramble ──────────────
ejemplo, moves_aplicados = scramble_cube(n_moves=5, seed=42)
print(f"Movimientos de scramble: {moves_aplicados}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#1e1e2e')
for ax in axes:
    ax.set_facecolor('#1e1e2e')

draw_cube_2x2(SOLVED_STATE, ax=axes[0], title="Estado Inicial (Resuelto)")
draw_cube_2x2(ejemplo, ax=axes[1], title=f"Después de {len(moves_aplicados)} movimientos")

plt.suptitle("Scramble del Cubo Rubik 2×2", fontsize=13, fontweight='bold',
             color='#e0e0e0', y=1.01)
plt.tight_layout()
plt.show()

## 5. 🧮 Heurística Admisible
Usamos la heurística combinada: `max(stickers_mal_ubicados/8, cubitos_mal_ubicados/4)`. Es admisible porque nunca sobreestima el costo real.

In [ ]:
def heuristic(state):
    """
    Heurística admisible combinada.
    h1: stickers mal ubicados / 8  (max 8 stickers cambian por movimiento)
    h2: cubitos mal ubicados  / 4  (max 4 cubitos cambian por movimiento)
    Retorna max(h1, h2) para ser lo más informada posible.
    """
    # h1: stickers fuera de posición
    h1 = sum(1 for i in range(24) if state[i] != SOLVED_STATE[i]) / 8.0

    # h2: cubitos de esquina mal ubicados (el 2x2 tiene 8 esquinas)
    CORNERS = [(0,8,16),(1,9,20),(2,13,16),(3,12,20),
                (4,10,18),(5,11,22),(6,15,18),(7,14,22)]
    wrong = sum(1 for c in CORNERS
                if any(state[i] != SOLVED_STATE[i] for i in c))
    h2 = wrong / 4.0

    return max(h1, h2)

print(f"h(resuelto)         = {heuristic(SOLVED_STATE)}  (esperado: 0.0)")
test = MOVES['U'](MOVES['R'](SOLVED_STATE))
print(f"h(2 movimientos)    = {heuristic(test)}  (esperado: > 0)")
print(f"h(5 mov. scramble)  = {heuristic(ejemplo)}")
print("✅ Heurística lista.")

## 6. 🔍 Implementación de A* y Weighted A*

- **A\*** (`weight=1.0`): `f(n) = g(n) + h(n)` → solución óptima garantizada.
- **Weighted A\*** (`weight>1.0`): `f(n) = g(n) + w·h(n)` → más rápido, suboptimalidad acotada en `w·C*`.

In [ ]:
def astar(start_state, weight=1.0, time_limit=30):
    """
    A* generalizado para resolver el Cubo Rubik 2x2.

    weight=1.0  → A* clásico (solución óptima)
    weight>1.0  → Weighted A* (más rápido, solución ≤ w·óptima)

    Retorna dict: solution, nodes_expanded, time, found, solution_length
    """
    if is_solved(start_state):
        return {'solution':[], 'nodes_expanded':0, 'time':0,
                'found':True, 'solution_length':0}

    t0 = time.time()
    # Cola: (f, g, estado, camino)
    heap = [(weight * heuristic(start_state), 0, start_state, [])]
    closed = {}   # estado → menor g visto
    expanded = 0

    while heap:
        if time.time() - t0 > time_limit:
            return {'solution':None, 'nodes_expanded':expanded,
                    'time':time.time()-t0, 'found':False, 'solution_length':-1}

        f, g, state, path = heapq.heappop(heap)

        if state in closed and closed[state] <= g:
            continue
        closed[state] = g
        expanded += 1

        for move_name, move_fn in MOVES.items():
            ns = move_fn(state)
            ng = g + 1
            if ns in closed and closed[ns] <= ng:
                continue
            if is_solved(ns):
                return {'solution': path + [move_name],
                        'nodes_expanded': expanded,
                        'time': time.time() - t0,
                        'found': True,
                        'solution_length': len(path) + 1}
            nf = ng + weight * heuristic(ns)
            heapq.heappush(heap, (nf, ng, ns, path + [move_name]))

    return {'solution':None, 'nodes_expanded':expanded,
            'time':time.time()-t0, 'found':False, 'solution_length':-1}


def verify_solution(start, moves_list):
    """Aplica los movimientos al estado inicial y verifica que quede resuelto."""
    s = start
    for m in moves_list:
        s = MOVES[m](s)
    return is_solved(s)

print("✅ Algoritmo A* / Weighted A* implementado.")

## 7. 🧪 Prueba de Verificación con Visualización Paso a Paso

In [ ]:
# ── Resolver un cubo de 4 movimientos con A* y Weighted A* ────
cubo_test, scramble_test = scramble_cube(n_moves=4, seed=7)
print(f"Scramble: {scramble_test}")

res_a  = astar(cubo_test, weight=1.0)
res_wa = astar(cubo_test, weight=2.0)

print(f"\n{'Algoritmo':<20} {'Tiempo':>10} {'Nodos':>10} {'Solución'}")
print("-" * 65)
print(f"{'A* (w=1.0)':<20} {res_a['time']:>9.4f}s {res_a['nodes_expanded']:>10,}  {res_a['solution']}")
print(f"{'Weighted A* (w=2)':<20} {res_wa['time']:>9.4f}s {res_wa['nodes_expanded']:>10,}  {res_wa['solution']}")

assert verify_solution(cubo_test, res_a['solution']),  "❌ A* no resolvió el cubo"
assert verify_solution(cubo_test, res_wa['solution']), "❌ Weighted A* no resolvió el cubo"
print("\n✅ Ambas soluciones verificadas correctamente.")

# ── Visualizar la solución paso a paso (A*) ───────────────────
steps_to_show = min(4, len(res_a['solution']))
n_panels = steps_to_show + 1  # inicial + pasos

fig, axes = plt.subplots(1, n_panels, figsize=(n_panels * 7, 5))
fig.patch.set_facecolor('#1e1e2e')
for ax in axes:
    ax.set_facecolor('#1e1e2e')

state = cubo_test
draw_cube_2x2(state, ax=axes[0], title="Estado inicial")

for i, move in enumerate(res_a['solution'][:steps_to_show]):
    state = MOVES[move](state)
    label = f"Paso {i+1}: {move}"
    if is_solved(state):
        label += " ✅"
    draw_cube_2x2(state, ax=axes[i+1], title=label)

plt.suptitle(f"Solución A* paso a paso  ({res_a['solution_length']} movimientos)",
             fontsize=13, fontweight='bold', color='#e0e0e0', y=1.01)
plt.tight_layout()
plt.show()

## 8. 📊 Experimento Comparativo: A* vs Weighted A*
Evaluamos múltiples niveles de dificultad (2–7 movimientos de scramble) con distintos valores de `w`.

In [ ]:
# ── Configuración ─────────────────────────────────────────────
SCRAMBLE_LEVELS = [2, 3, 4, 5, 6, 7]
N_TRIALS  = 5
WEIGHTS   = [1.0, 1.5, 2.0, 3.0]
TIME_LIMIT = 20

results = {w: {'times':[], 'nodes':[], 'lengths':[], 'levels':[]}
           for w in WEIGHTS}

total = len(SCRAMBLE_LEVELS) * N_TRIALS
done  = 0

for level in SCRAMBLE_LEVELS:
    times_w  = {w: [] for w in WEIGHTS}
    nodes_w  = {w: [] for w in WEIGHTS}
    length_w = {w: [] for w in WEIGHTS}

    for trial in range(N_TRIALS):
        seed = level * 100 + trial
        cubo, _ = scramble_cube(n_moves=level, seed=seed)
        for w in WEIGHTS:
            res = astar(cubo, weight=w, time_limit=TIME_LIMIT)
            times_w[w].append(res['time'])
            nodes_w[w].append(res['nodes_expanded'])
            if res['found']:
                length_w[w].append(res['solution_length'])
        done += 1
        print(f"\r  ⏳ Progreso: {done}/{total} ({100*done//total}%)   ", end='')

    for w in WEIGHTS:
        results[w]['times'].append(np.mean(times_w[w]))
        results[w]['nodes'].append(np.mean(nodes_w[w]))
        results[w]['levels'].append(level)
        results[w]['lengths'].append(np.mean(length_w[w]) if length_w[w] else None)

print("\n\n✅ Experimento completo.")

## 9. 📈 Visualización de Resultados

In [ ]:
# ══════════════════════════════════════════════════════════════
#  GRÁFICA 1: Tiempo | Nodos | Longitud de solución
# ══════════════════════════════════════════════════════════════

PLOT_COLORS = {
    1.0: '#E74C3C',
    1.5: '#F39C12',
    2.0: '#2ECC71',
    3.0: '#3498DB',
}
LABELS = {
    1.0: 'A* clásico (w=1.0)',
    1.5: 'Weighted A* (w=1.5)',
    2.0: 'Weighted A* (w=2.0)',
    3.0: 'Weighted A* (w=3.0)',
}
MARKERS = {1.0:'o', 1.5:'s', 2.0:'^', 3.0:'D'}

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.patch.set_facecolor('#1e1e2e')
for ax in axes:
    ax.set_facecolor('#2a2a3e')
    ax.tick_params(colors='#cccccc')
    ax.xaxis.label.set_color('#cccccc')
    ax.yaxis.label.set_color('#cccccc')
    ax.title.set_color('#e0e0e0')
    for spine in ax.spines.values():
        spine.set_edgecolor('#444444')

fig.suptitle('Comparación A* vs Weighted A* — Cubo Rubik 2×2',
             fontsize=15, fontweight='bold', color='#ffffff', y=1.02)

# ── Tiempo ────────────────────────────────────────────────────
for w in WEIGHTS:
    axes[0].plot(results[w]['levels'], results[w]['times'],
                 color=PLOT_COLORS[w], marker=MARKERS[w],
                 linewidth=2.5, markersize=8, label=LABELS[w])
axes[0].set_xlabel('Movimientos de Scramble')
axes[0].set_ylabel('Tiempo promedio (s)')
axes[0].set_title('⏱️  Tiempo de Ejecución')
axes[0].set_yscale('log')
axes[0].legend(fontsize=8, facecolor='#2a2a3e', labelcolor='#cccccc')
axes[0].grid(True, alpha=0.2, color='#888888')
axes[0].set_xticks(SCRAMBLE_LEVELS)

# ── Nodos Expandidos ──────────────────────────────────────────
for w in WEIGHTS:
    axes[1].plot(results[w]['levels'], results[w]['nodes'],
                 color=PLOT_COLORS[w], marker=MARKERS[w],
                 linewidth=2.5, markersize=8, label=LABELS[w])
axes[1].set_xlabel('Movimientos de Scramble')
axes[1].set_ylabel('Nodos expandidos (promedio)')
axes[1].set_title('🔍 Nodos Explorados')
axes[1].set_yscale('log')
axes[1].legend(fontsize=8, facecolor='#2a2a3e', labelcolor='#cccccc')
axes[1].grid(True, alpha=0.2, color='#888888')
axes[1].set_xticks(SCRAMBLE_LEVELS)

# ── Longitud de Solución ──────────────────────────────────────
for w in WEIGHTS:
    lvls = [l for l, le in zip(results[w]['levels'], results[w]['lengths']) if le]
    lens = [le for le in results[w]['lengths'] if le]
    axes[2].plot(lvls, lens, color=PLOT_COLORS[w], marker=MARKERS[w],
                 linewidth=2.5, markersize=8, label=LABELS[w])
axes[2].set_xlabel('Movimientos de Scramble')
axes[2].set_ylabel('Longitud de solución (movimientos)')
axes[2].set_title('📏 Calidad de Solución')
axes[2].legend(fontsize=8, facecolor='#2a2a3e', labelcolor='#cccccc')
axes[2].grid(True, alpha=0.2, color='#888888')
axes[2].set_xticks(SCRAMBLE_LEVELS)

plt.tight_layout()
plt.savefig('comparacion_astar_rubik.png', dpi=150, bbox_inches='tight',
            facecolor='#1e1e2e')
plt.show()
print("✅ Guardado: comparacion_astar_rubik.png")

In [ ]:
# ══════════════════════════════════════════════════════════════
#  GRÁFICA 2: Speedup y Suboptimalidad
# ══════════════════════════════════════════════════════════════

fig2, (ax_sp, ax_sub) = plt.subplots(1, 2, figsize=(14, 6))
fig2.patch.set_facecolor('#1e1e2e')
for ax in (ax_sp, ax_sub):
    ax.set_facecolor('#2a2a3e')
    ax.tick_params(colors='#cccccc')
    ax.xaxis.label.set_color('#cccccc')
    ax.yaxis.label.set_color('#cccccc')
    ax.title.set_color('#e0e0e0')
    for spine in ax.spines.values():
        spine.set_edgecolor('#444444')

fig2.suptitle('Speedup y Costo de Suboptimalidad del Weighted A*',
              fontsize=14, fontweight='bold', color='#ffffff', y=1.02)

base_times   = results[1.0]['times']
base_lengths = results[1.0]['lengths']

# ── Speedup ───────────────────────────────────────────────────
for w in [1.5, 2.0, 3.0]:
    sp = [bt/wt if wt > 0 else 1
          for bt, wt in zip(base_times, results[w]['times'])]
    ax_sp.plot(SCRAMBLE_LEVELS, sp,
               color=PLOT_COLORS[w], marker=MARKERS[w],
               linewidth=2.5, markersize=8, label=LABELS[w])

ax_sp.axhline(1, color='#E74C3C', linestyle='--', alpha=0.6,
              label='A* base (speedup=1)')
ax_sp.set_xlabel('Movimientos de Scramble')
ax_sp.set_ylabel('Speedup (× más rápido que A*)')
ax_sp.set_title('⚡ Speedup del Weighted A*')
ax_sp.legend(fontsize=9, facecolor='#2a2a3e', labelcolor='#cccccc')
ax_sp.grid(True, alpha=0.2, color='#888888')
ax_sp.set_xticks(SCRAMBLE_LEVELS)

# ── Suboptimalidad ────────────────────────────────────────────
for w in [1.5, 2.0, 3.0]:
    overhead = []
    for bl, wl in zip(base_lengths, results[w]['lengths']):
        if bl and wl and bl > 0:
            overhead.append(100 * (wl - bl) / bl)
        else:
            overhead.append(0)
    ax_sub.plot(SCRAMBLE_LEVELS, overhead,
                color=PLOT_COLORS[w], marker=MARKERS[w],
                linewidth=2.5, markersize=8, label=LABELS[w])

ax_sub.axhline(0, color='#2ECC71', linestyle='--', alpha=0.6,
               label='Óptimo (0%)')
ax_sub.set_xlabel('Movimientos de Scramble')
ax_sub.set_ylabel('Suboptimalidad (%)')
ax_sub.set_title('📉 Costo de Suboptimalidad')
ax_sub.legend(fontsize=9, facecolor='#2a2a3e', labelcolor='#cccccc')
ax_sub.grid(True, alpha=0.2, color='#888888')
ax_sub.set_xticks(SCRAMBLE_LEVELS)

plt.tight_layout()
plt.savefig('speedup_suboptimalidad.png', dpi=150, bbox_inches='tight',
            facecolor='#1e1e2e')
plt.show()
print("✅ Guardado: speedup_suboptimalidad.png")

## 10. 📋 Tabla Resumen

In [ ]:
print("=" * 80)
print(f"{'COMPARACIÓN A* vs Weighted A*  —  Cubo Rubik 2×2':^80}")
print("=" * 80)
print(f"{'Scramble':^8} {'Algoritmo':^22} {'Tiempo(s)':^11} {'Nodos':^12} {'Long.Sol.':^10}")
print("-" * 80)

for i, level in enumerate(SCRAMBLE_LEVELS):
    for w in WEIGHTS:
        tag = " [óptimo]" if w == 1.0 else ""
        name = f"A* w={w:.1f}{tag}"
        t = results[w]['times'][i]
        n = int(results[w]['nodes'][i])
        l = results[w]['lengths'][i]
        ls = f"{l:.1f}" if l else "N/A"
        print(f"{level:^8} {name:^22} {t:^11.4f} {n:^12,} {ls:^10}")
    print("-" * 80)

# Resumen estadístico
speedups = [bt/wt for bt, wt in zip(results[1.0]['times'], results[2.0]['times']) if wt>0]
overheads = [100*(wl-bl)/bl for bl, wl in zip(results[1.0]['lengths'], results[2.0]['lengths'])
             if bl and wl and bl > 0]

print(f"\n📌 Weighted A* (w=2.0) es en promedio {np.mean(speedups):.1f}× más rápido que A* clásico")
if overheads:
    print(f"   con una suboptimalidad promedio de solo {np.mean(overheads):.1f}%")

## 11. 🎬 Demo Final: Solución Completa Visualizada

In [ ]:
# ── Resolver un cubo de 5 movimientos y mostrar comparación ───
demo_cubo, demo_scramble = scramble_cube(n_moves=5, seed=99)
print(f"🎲 Scramble: {demo_scramble}")

res_final_a  = astar(demo_cubo, weight=1.0)
res_final_wa = astar(demo_cubo, weight=2.0)

print(f"\nA* clásico:     {res_final_a['solution_length']} movimientos en {res_final_a['time']:.4f}s  ({res_final_a['nodes_expanded']:,} nodos)")
print(f"Weighted A*:    {res_final_wa['solution_length']} movimientos en {res_final_wa['time']:.4f}s  ({res_final_wa['nodes_expanded']:,} nodos)")

# ── Visualizar: scramble + solución A* (primeros 3 pasos) ─────
n_show  = min(3, res_final_a['solution_length'])
n_panels = 1 + n_show

fig, axes = plt.subplots(1, n_panels, figsize=(n_panels * 7, 5))
if n_panels == 1:
    axes = [axes]
fig.patch.set_facecolor('#1e1e2e')
for ax in axes:
    ax.set_facecolor('#1e1e2e')

draw_cube_2x2(demo_cubo, ax=axes[0], title="Cubo a resolver")

state = demo_cubo
for i, move in enumerate(res_final_a['solution'][:n_show]):
    state = MOVES[move](state)
    lbl = f"Paso {i+1}: {move}" + (" ✅" if is_solved(state) else "")
    draw_cube_2x2(state, ax=axes[i+1], title=lbl)

plt.suptitle(
    f"A* → {res_final_a['solution_length']} movs  |  Weighted A* → {res_final_wa['solution_length']} movs",
    fontsize=13, fontweight='bold', color='#e0e0e0', y=1.01
)
plt.tight_layout()
plt.show()
print("\n🏁 ¡Experimento completado exitosamente!")